# Day 7 - 작은 fine-tuning

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_7
- paper_ids: qwen3_vl_2025, lora_2021
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

작은 VLM dataset으로 overfit·leakage·format memorization을 피하면서 LoRA SFT 한 사이클을 어떻게 설계하고 중단할 것인가?

## 2. Background theory

SFT는 image+user prompt를 조건으로 assistant token의 next-token loss를 최소화한다. prompt/image token label은 -100으로 mask하고 assistant answer만 loss에 포함한다. 비슷한 video frame이나 같은 document page는 group 단위로 split한다.

## 3. Paper connection

Qwen3-VL 공식 training framework는 vision/projector/LLM tune flag, resolution, LoRA rank/alpha를 분리한다. 이 과정은 2B Instruct, batch 1, 20-step smoke run을 기본으로 하며 성능 claim이 아닌 pipeline 검증이다.

## 4. Input/output and shapes

JSONL 한 줄은 `image,user,assistant,group_id`다. image는 data root 상대 경로다. tokenized full conversation `[1,L]`, labels `[1,L]`, loss는 assistant 구간만 계산한다. output은 LoRA adapter와 processor config다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

synthetic metadata를 group split해 같은 source가 train/validation 양쪽에 들어가지 않는지 확인한다.

In [ ]:
from vlm_foundation.sft import SftExample, group_split

examples = [
    SftExample(f"images/{group}_{i}.jpg", "Locate the helmet.", "[]", group)
    for group in ["camera_a_clip_1", "camera_a_clip_2", "camera_b_clip_1"]
    for i in range(3)
]
train, validation = group_split(examples, validation_fraction=0.34, seed=42)
print("train groups:", sorted({x.group_id for x in train}))
print("validation groups:", sorted({x.group_id for x in validation}))

## 6. Visualization sanity check

학습 전 실제 sample의 image, prompt, answer, token 길이, label mask를 한 건 확인한다. 잘못된 answer format을 대량 학습하기 전에 멈춘다.

In [ ]:
assert {x.group_id for x in train}.isdisjoint({x.group_id for x in validation})
print("group leakage check: PASS")

## 7. Experiment

1) base zero-shot, 2) 20-step overfit smoke, 3) 작은 train run을 구분한다. 각 단계에서 validation과 고정 qualitative set을 비교한다.

In [ ]:
print("""Dry-run command:
PYTHONPATH=src python scripts/train_lora.py \
  --data-root /nas/datahub/min/<user-selected> \
  --manifest manifests/tiny_sft.jsonl

GPU training starts only when --execute is added.
Missing model files are never downloaded unless --allow-download is also added.
""")

## 8. Metrics

train/validation loss, exact JSON validity, task IoU/accuracy, held-out group 성능, base prompt 회귀, peak memory, tokens/sec를 기록한다.

## 9. Interpretation

train loss 하락만으로 visual grounding을 학습했다고 결론내리지 않는다. image를 바꾸어도 같은 답을 내면 language-format memorization일 수 있다.

## 10. Failure cases

group leakage, answer-only shortcut, malformed JSON, image path mismatch, assistant mask off-by-one, OOM, catastrophic forgetting, validation prompt tuning을 확인한다.

## 11. Real-service implications

adapter는 base revision과 함께 배포하고 A/B shadow evaluation을 거친다. training data license, PII, failure examples, rollback artifact를 보존한다.

## 12. Review questions

1. 왜 prompt token label을 -100으로 mask하는가?
2. frame random split이 leakage를 만드는 이유는?
3. 20-step smoke run의 목적은?
4. format memorization과 visual learning을 어떻게 구분하는가?
5. 실제 실행 전에 지정할 data root와 manifest는?